In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

In [2]:
df = pd.read_csv("../data/contextual_merged_dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 21)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,timestamp,Timestamp,Ambient_Temperature,Load_Density,Humidity,Shift,Day_Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2025-01-01 00:00:00,2025-01-01 00:00:00,26,62,41,Evening,Weekday
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2025-01-01 00:01:00,2025-01-01 00:01:00,39,34,47,Morning,Weekend
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2025-01-01 00:02:00,2025-01-01 00:02:00,34,43,81,Evening,Weekday
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2025-01-01 00:03:00,2025-01-01 00:03:00,30,97,46,Evening,Weekend
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2025-01-01 00:04:00,2025-01-01 00:04:00,27,76,49,Morning,Weekday


In [3]:
target = "Machine failure"

X = df.drop(columns=[target])
y = df[target]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

print("\nOriginal Class Distribution:")
print(y.value_counts())


Original Class Distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [4]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified CV Initialized")

5-Fold Stratified CV Initialized


In [5]:
fold_summary = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_dist = y_train.value_counts()
    test_dist = y_test.value_counts()

    print(f"\n========== Fold {fold} ==========")

    print("\nTraining Distribution:")
    print(train_dist)

    print("\nTesting Distribution:")
    print(test_dist)

    fold_summary.append({
        "Fold": fold,
        "Train_Normal": train_dist.get(0, 0),
        "Train_Failure": train_dist.get(1, 0),
        "Test_Normal": test_dist.get(0, 0),
        "Test_Failure": test_dist.get(1, 0)
    })


========== Fold 1 ==========

Training Distribution:
Machine failure
0    7728
1     272
Name: count, dtype: int64

Testing Distribution:
Machine failure
0    1933
1      67
Name: count, dtype: int64

========== Fold 2 ==========

Training Distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing Distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64

========== Fold 3 ==========

Training Distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing Distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64

========== Fold 4 ==========

Training Distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing Distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64

========== Fold 5 ==========

Training Distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing Distribution:
Machine failure
0    1932
1      68
Name: count, dtype: i

In [6]:
fold_df = pd.DataFrame(fold_summary)

print("\nFold-wise Distribution Summary")
fold_df


Fold-wise Distribution Summary


,Fold,Train_Normal,Train_Failure,Test_Normal,Test_Failure
0,1,7728,272,1933,67
1,2,7729,271,1932,68
2,3,7729,271,1932,68
3,4,7729,271,1932,68
4,5,7729,271,1932,68


In [7]:
print("\nVerification Complete")

for i in range(len(fold_df)):
    print(
        f"Fold {i+1}: "
        f"Train Ratio = "
        f"{fold_df.iloc[i]['Train_Failure']}/"
        f"{fold_df.iloc[i]['Train_Normal']}, "
        f"Test Ratio = "
        f"{fold_df.iloc[i]['Test_Failure']}/"
        f"{fold_df.iloc[i]['Test_Normal']}"
    )

print("\nAll folds preserve class distribution successfully.")


Verification Complete
Fold 1: Train Ratio = 272/7728, Test Ratio = 67/1933
Fold 2: Train Ratio = 271/7729, Test Ratio = 68/1932
Fold 3: Train Ratio = 271/7729, Test Ratio = 68/1932
Fold 4: Train Ratio = 271/7729, Test Ratio = 68/1932
Fold 5: Train Ratio = 271/7729, Test Ratio = 68/1932

All folds preserve class distribution successfully.
